# RawTree · consola de requests
Ejecuta la **celda maestra** una vez. Después cualquier celda funciona por separado, sin ejecutar las anteriores. Cada celda expone su SQL y sus parámetros locales, como una petición en Postman. Solo se hacen lecturas; nada se publica en el leaderboard.

Los campos siguen el esquema del README del reto; no se ha verificado el esquema remoto. Las tablas del stream vacías pueden devolver `Unknown identifier`. Los límites restringen resultados, no necesariamente el coste de lectura. Tiempos en UTC. No compartas salidas con datos privados ni el archivo `.env`.


In [1]:
# CELDA MAESTRA: dependencias, credenciales y cliente HTTP.
import importlib.util
import subprocess
import sys
import shutil
missing = [name for name in ("requests", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    # uv crea entornos sin pip por defecto: instala en el Python del kernel.
    if shutil.which("uv"):
        command = ["uv", "pip", "install", "--python", sys.executable, *missing]
    elif importlib.util.find_spec("pip") is not None:
        command = [sys.executable, "-m", "pip", "install", *missing]
    else:
        raise RuntimeError("Faltan " + ", ".join(missing)
                           + ". Instálalas con uv pip install --python " + sys.executable
                           + " " + " ".join(missing))
    subprocess.run(command, check=True)
    importlib.invalidate_caches()

import os
import re
import time
import uuid
from pathlib import Path
from getpass import getpass
import pandas as pd
import requests
from IPython.display import display

# Busca .env desde el directorio del kernel hacia arriba; entorno tiene prioridad.
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / ".env").exists():
        # Formato KEY=value; admite comentarios, export y comillas externas.
        # No ejecuta shell ni expande variables.
        for raw_line in (directory / ".env").read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].lstrip()
            key, separator, value = line.partition("=")
            if not separator or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key.strip()):
                continue
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
                value = value[1:-1]
            else:
                value = re.split(r"\s+#", value, maxsplit=1)[0].rstrip()
            os.environ.setdefault(key.strip(), value)
        break
API = os.getenv("RAWTREE_API_URL", "https://api.rawtree.com").rstrip("/")
TEAM = os.getenv("TEAM_ID") or input("TEAM_ID (team_XX): ").strip()
assert re.fullmatch(r"team_[A-Za-z0-9_]+", TEAM), "TEAM_ID inválido"
KEY = os.getenv("RAWTREE_API_KEY") or getpass("RAWTREE_API_KEY: ")
HTTP = requests.Session()
HTTP.headers.update({"Authorization": f"Bearer {KEY}", "User-Agent": "datathon-notebook/1.0"})

def literal(value):
    """Literal SQL de texto; no usar para nombres de tabla/columna."""
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"

def query(sql, timeout=60):
    query_id = str(uuid.uuid4())
    start = time.monotonic()
    response = HTTP.post(API + "/v1/query", params={"database": TEAM},
                         json={"sql": sql, "format": "JSON", "query_id": query_id},
                         timeout=(10, timeout))
    print(f"POST /v1/query | HTTP {response.status_code} | {time.monotonic()-start:.2f}s | {query_id}")
    if not response.ok:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:2000]}")
    payload = response.json()
    print("rows:", payload.get("rows"), "statistics:", payload.get("statistics", {}))
    if payload.get("hints"):
        print("HINTS (revisar tipos omitidos):", payload["hints"])
    frame = pd.DataFrame(payload.get("data", []))
    display(frame)
    return frame

print("Cliente listo para", TEAM, "— credencial oculta")


Cliente listo para team_14 — credencial oculta


## 01 · Conexión


In [2]:

sql = f"""
SELECT 1 AS ok, now() AS server_utc
"""
result = query(sql)


POST /v1/query | HTTP 200 | 10.60s | a070e3cf-d6ee-4795-9097-aa1becdd52d5
rows: 1 statistics: {'elapsed': 0.001010281, 'rows_read': 1, 'bytes_read': 1}


,ok,server_utc
0,1,2026-11-14 12:33:51


## 02 · Esquema de una tabla


In [3]:
table = "shop.requests"  # Editar: shop.catalog, shop.events, {TEAM}.feedback
assert re.fullmatch(r"(?:shop|" + TEAM + r")\.[A-Za-z_]+", table)
sql = f"""
DESCRIBE TABLE {table} SETTINGS describe_include_subcolumns = 1
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.59s | 49e84c20-2156-4a8f-ba89-7ba185edf058
rows: 8 statistics: {'elapsed': 0.000868101, 'rows_read': 8, 'bytes_read': 594}


,name,type,default_type,default_expression,comment,codec_expression,ttl_expression,is_subcolumn
0,context.device,Dynamic,,,,,,1
1,context.page,Dynamic,,,,,,1
2,context.region,Dynamic,,,,,,1
3,is_new_user,Dynamic,,,,,,1
4,recent_views,Dynamic,,,,,,1
5,request_id,Dynamic,,,,,,1
6,ts,Dynamic,,,,,,1
7,user_id,Dynamic,,,,,,1


## 03 · Últimas peticiones


In [6]:

sql = f"""
SELECT request_id, ts, user_id, is_new_user, context, recent_views
FROM shop.requests ORDER BY ts::DateTime64(3) DESC, request_id::String LIMIT 30
"""
result = query(sql)


POST /v1/query | HTTP 200 | 4.05s | 08ca11b1-4671-4254-a6ae-dbb9cdef87ca
rows: 30 statistics: {'elapsed': 0.011048872, 'rows_read': 38952, 'bytes_read': 2380877}


,request_id,ts,user_id,is_new_user,context,recent_views
0,r_8d37d16c4006f69b,2026-11-14 12:34:54.338000000,u_951b32a1,False,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_b49cb8', 'ts': '2026-11-14 ..."
1,r_5915c33baf96f42b,2026-11-14 12:34:54.024000000,u_45eb9921,True,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_5063b4', 'ts': '2026-11-14 ..."
2,r_0399fcff436a32dc,2026-11-14 12:34:53.667000000,u_ece47f09,False,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_0b7297', 'ts': '2026-11-14 ..."
3,r_7b551b62fc300e1e,2026-11-14 12:34:53.359000000,u_d48c4234,False,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_ced957', 'ts': '2026-11-14 ..."
4,r_48d39edb43e0eb8a,2026-11-14 12:34:53.044000000,u_d1a0fa72,False,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_21e864', 'ts': '2026-11-14 ..."
5,r_9f593344f8bcc4a7,2026-11-14 12:34:52.688000000,u_7ff9838b,False,"{'device': 'ios', 'page': 'home', 'region': 'C...",[]
6,r_38ce7a3b64f84118,2026-11-14 12:34:52.360000000,u_4617768b,True,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_dc1c2e', 'ts': '2026-11-14 ..."
7,r_5f9b2c5e343d9f1f,2026-11-14 12:34:52.046000000,u_422ce5b0,False,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_0e2b03', 'ts': '2026-11-14 ..."
8,r_2ee21ad0632956b3,2026-11-14 12:34:51.689000000,u_e1f258dd,False,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_3b7af8', 'ts': '2026-11-14 ..."
9,r_880cdf642463f92c,2026-11-14 12:34:51.370000000,u_c5b8edea,False,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_4895df', 'ts': '2026-11-14 ..."


## 04 · Catálogo inicial


In [7]:

sql = f"""
SELECT product_id, category, subcategory, brand, price_eur, is_active
FROM shop.catalog ORDER BY product_id::String LIMIT 100
"""
result = query(sql)


POST /v1/query | HTTP 200 | 2.05s | aac9285b-7cd6-482c-8f06-6c4c9dbbaf37
rows: 100 statistics: {'elapsed': 0.005210239, 'rows_read': 10000, 'bytes_read': 689219}


,product_id,category,subcategory,brand,price_eur,is_active
0,p_001897,Sports & Outdoors,Yoga Mats,Vedo,14.99,True
1,p_00196b,Fashion,Jackets,Zore,4.99,True
2,p_001a20,Electronics,Laptop Sleeves,Sanine,57.99,True
3,p_001d7f,Sports & Outdoors,Running Shoes,Vebexa,29.99,True
4,p_0032c5,Home & Kitchen,Cutting Boards,Nibe,26.99,True
...,...,...,...,...,...,...
95,p_04b43f,Electronics,Phone Cases,Sanine,12.99,True
96,p_04be9c,Books & Media,E-Reader Covers,Radomi,239.99,True
97,p_04c49f,Computers & Gaming,Mice,Mariti,25.99,True
98,p_04cf5f,Pets,Fish Food,Lumi,39.99,True


## 05 · Catálogo actual
Última actualización completa sobre el catálogo inicial. Empates de timestamp no tienen desempate definido.

In [9]:

sql = f"""
SELECT product_id,
 argMax(category, version) AS category, argMax(subcategory, version) AS subcategory,
 argMax(price_eur, version) AS price_eur, argMax(is_active, version) AS is_active
FROM (
 SELECT product_id::String AS product_id, category::String AS category,
 subcategory::String AS subcategory, price_eur::Float64 AS price_eur,
 is_active::Bool AS is_active, created_at::DateTime64(3) AS version FROM shop.catalog
 UNION ALL
 SELECT product_id::String, category::String, subcategory::String,
 price_eur::Float64, is_active::Bool, ts::DateTime64(3) FROM shop.catalog_updates
) GROUP BY product_id ORDER BY product_id LIMIT 6000
"""
result = query(sql)


POST /v1/query | HTTP 200 | 3.12s | 53f1e562-78cf-4562-ad40-eb64082b1684
rows: 5100 statistics: {'elapsed': 0.01307303, 'rows_read': 6575, 'bytes_read': 808792}


,product_id,category,subcategory,price_eur,is_active
0,p_00183c,Computers & Gaming,External Drives,31.99,True
1,p_001897,Sports & Outdoors,Yoga Mats,14.99,True
2,p_00196b,Fashion,Jackets,4.99,True
3,p_001a20,Electronics,Laptop Sleeves,57.99,True
4,p_001d7f,Sports & Outdoors,Running Shoes,29.99,True
...,...,...,...,...,...
5095,p_ffacff,Home & Garden,Lamps,17.99,True
5096,p_ffb2c5,Electronics,Laptops,2.99,True
5097,p_ffcfa5,Sports & Outdoors,Tents,38.99,True
5098,p_ffdd4e,Toys & Baby,Building Blocks,42.99,True


## 06 · Cambios de stock y precio


In [10]:

sql = f"""
SELECT product_id, ts, change, price_eur, is_active
FROM shop.catalog_updates ORDER BY ts::DateTime64(3) DESC, product_id::String LIMIT 50
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.25s | 611cddf1-5f5b-4ac0-aa9a-b6190db726a6
rows: 50 statistics: {'elapsed': 0.003845827, 'rows_read': 3150, 'bytes_read': 166949}


,product_id,ts,change,price_eur,is_active
0,p_188733,2026-11-14 12:40:00.000000000,back_in_stock,25.99,True
1,p_1d2fae,2026-11-14 12:40:00.000000000,back_in_stock,7.99,True
2,p_3d987d,2026-11-14 12:40:00.000000000,out_of_stock,11.99,False
3,p_40bbb4,2026-11-14 12:40:00.000000000,out_of_stock,330.99,False
4,p_9a292a,2026-11-14 12:40:00.000000000,out_of_stock,21.99,False
5,p_d7db9b,2026-11-14 12:40:00.000000000,back_in_stock,24.99,True
6,p_01e69f,2026-11-14 12:35:00.000000000,back_in_stock,96.99,True
7,p_578480,2026-11-14 12:35:00.000000000,back_in_stock,31.99,True
8,p_5eb51d,2026-11-14 12:35:00.000000000,back_in_stock,4.99,True
9,p_d19912,2026-11-14 12:35:00.000000000,out_of_stock,32.99,False


## 07 · Producto concreto


In [13]:
product_id = "p_06e168"
sql = f"""
SELECT product_id, category, subcategory, brand, price_eur, description, is_active
FROM shop.catalog WHERE product_id::String = {literal(product_id)} LIMIT 1
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.35s | 9536eecb-c529-4c31-9c3c-a739dcd99e85
rows: 1 statistics: {'elapsed': 0.005444894, 'rows_read': 5000, 'bytes_read': 1093742}


,product_id,category,subcategory,brand,price_eur,description,is_active
0,p_06e168,Pets,Pet Shampoo,Ririmi,38.99,"Pet Shampoo by Ririmi, featuring organic ingre...",True


## 08 · Actividad reciente


In [12]:
minutes = 15
assert isinstance(minutes, int) and 1 <= minutes <= 1440
sql = f"""
SELECT ts, user_id, session_id, product_id, event_type, price_eur
FROM shop.events WHERE ts::DateTime64(3) > now() - INTERVAL {minutes} MINUTE
ORDER BY ts::DateTime64(3) DESC, product_id::String LIMIT 100
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.37s | 52a9c3c8-6ecc-400b-b463-329d5b6d480b
rows: 100 statistics: {'elapsed': 0.011216937, 'rows_read': 67914, 'bytes_read': 3401342}


,ts,user_id,session_id,product_id,event_type,price_eur
0,2026-11-14 12:42:37.971000000,u_24a34a20,s_8df1137f6f4e,p_b227ec,view,114.99
1,2026-11-14 12:42:37.964000000,u_24a34a20,s_8df1137f6f4e,p_e221fb,view,183.99
2,2026-11-14 12:42:37.964000000,u_24a34a20,s_8df1137f6f4e,p_e221fb,add_to_cart,183.99
3,2026-11-14 12:42:37.964000000,u_24a34a20,s_8df1137f6f4e,p_e221fb,purchase,183.99
4,2026-11-14 12:42:37.917000000,u_d1a51d8b,s_a2e01404c015,p_05a101,view,7.99
...,...,...,...,...,...,...
95,2026-11-14 12:42:36.474000000,u_7b8dd6be,s_772914743241,p_47cf9b,view,23.99
96,2026-11-14 12:42:36.431000000,u_7f88047c,s_9a6ca370eede,p_6e781c,view,90.99
97,2026-11-14 12:42:36.424000000,u_7f88047c,s_9a6ca370eede,p_00183c,view,31.99
98,2026-11-14 12:42:36.417000000,u_7f88047c,s_9a6ca370eede,p_e2454f,view,4.99


## 09 · Productos populares por categoría


In [14]:

sql = f"""
SELECT c.category AS category, e.product_id AS product_id,
 countIf(e.event_type = 'view') AS views, countIf(e.event_type = 'purchase') AS purchases
FROM (SELECT product_id::String AS product_id, event_type::String AS event_type
 FROM shop.events WHERE product_id IS NOT NULL AND ts::DateTime64(3) > now() - INTERVAL 1 HOUR) e
INNER JOIN (SELECT product_id::String AS product_id, category::String AS category
 FROM shop.catalog WHERE product_id IS NOT NULL) c ON e.product_id = c.product_id
GROUP BY category, product_id ORDER BY purchases DESC, views DESC, product_id LIMIT 5 BY category
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.40s | 1ba0aded-8716-4e64-a2e5-f9ad41ffb3a0
rows: 50 statistics: {'elapsed': 0.028115667, 'rows_read': 251112, 'bytes_read': 15978508}


,category,product_id,views,purchases
0,Books & Media,p_adc3e6,323,105
1,Toys & Baby,p_0210c5,306,92
2,Fashion,p_69d40e,252,87
3,Computers & Gaming,p_e2454f,291,80
4,Toys & Baby,p_1036d3,258,77
5,Electronics,p_2c88d5,234,57
6,Books & Media,p_032d24,210,52
7,Books & Media,p_989e85,207,52
8,Fashion,p_cbcc44,217,49
9,Electronics,p_8ca8ed,178,49


## 10 · Tendencias por subcategoría


In [15]:

sql = f"""
SELECT c.subcategory AS subcategory,
 countIf(e.event_ts > now() - INTERVAL 15 MINUTE) AS views_15m,
 countIf(e.event_ts <= now() - INTERVAL 15 MINUTE) / 12.0 AS previous_15m_average,
 views_15m / greatest(previous_15m_average, 1) AS lift
FROM (SELECT product_id::String AS product_id, ts::DateTime64(3) AS event_ts
 FROM shop.events WHERE product_id IS NOT NULL AND event_type = 'view'
 AND ts::DateTime64(3) > now() - INTERVAL 195 MINUTE) e
INNER JOIN (SELECT product_id::String AS product_id, subcategory::String AS subcategory
 FROM shop.catalog WHERE product_id IS NOT NULL) c ON e.product_id = c.product_id
GROUP BY subcategory ORDER BY lift DESC, subcategory LIMIT 20
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.23s | d5b01942-a063-4e2f-903d-9d0bce30d812
rows: 20 statistics: {'elapsed': 0.043465375, 'rows_read': 789244, 'bytes_read': 50305218}


,subcategory,views_15m,previous_15m_average,lift
0,Sunglasses,141,101.083333,1.394889
1,Sunscreen,285,209.500000,1.360382
2,Toy Batteries,253,193.916667,1.304684
3,Backpacks,261,212.000000,1.231132
4,Mice,256,208.000000,1.230769
5,Fitness Trackers,570,465.500000,1.224490
6,Sleeping Bags,894,761.583333,1.173870
7,Video Games,772,666.333333,1.158579
8,Shampoo,821,709.500000,1.157153
9,E-Readers,1071,931.250000,1.150067


## 11 · Productos vistos en las mismas sesiones
Deduplica por usuario, sesión y producto: cuenta sesiones, no pares de eventos repetidos. Puede ser costosa.

In [17]:
product_id = "p_3d987d"
start = "2026-11-01"
end = "2026-11-08"
sql = f"""
WITH session_products AS (
 SELECT DISTINCT user_id::String AS uid, session_id::String AS sid, product_id::String AS pid
 FROM shop.history_events
 WHERE event_type = 'view' AND user_id IS NOT NULL AND session_id IS NOT NULL
 AND product_id IS NOT NULL AND ts::DateTime64(3) >= {literal(start)}
 AND ts::DateTime64(3) < {literal(end)}
)
SELECT b.pid AS also_viewed, count() AS sessions
FROM (SELECT uid, sid, pid FROM session_products WHERE pid = {literal(product_id)}) a
INNER JOIN session_products b ON a.uid = b.uid AND a.sid = b.sid
WHERE b.pid != a.pid GROUP BY also_viewed ORDER BY sessions DESC, also_viewed LIMIT 20
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.69s | 24a03c53-ccbe-42db-8e89-9dd444c24d91
rows: 20 statistics: {'elapsed': 0.449760347, 'rows_read': 2883584, 'bytes_read': 351282290}


,also_viewed,sessions
0,p_b40dc3,23
1,p_125f5b,20
2,p_11f2a9,19
3,p_7c1929,18
4,p_f27fa2,16
5,p_29124c,15
6,p_ef7cd4,15
7,p_84c44c,14
8,p_f4465b,14
9,p_fb55b8,14


## 12 · Feedback reciente


In [18]:

sql = f"""
SELECT request_id, ts, served_by, fallback_reason, latency_ms, shown, clicked, purchased, revenue_eur
FROM {TEAM}.feedback WHERE request_id != '__init__'
ORDER BY ts::DateTime64(3) DESC, request_id::String LIMIT 50
"""
result = query(sql)


POST /v1/query | HTTP 200 | 0.39s | 4d697306-2ce1-4a24-a6d4-53683eb249cc
rows: 50 statistics: {'elapsed': 0.013369938, 'rows_read': 32124, 'bytes_read': 3764809}


,request_id,ts,served_by,fallback_reason,latency_ms,shown,clicked,purchased,revenue_eur
0,r_17c238a377dedcc3,2026-11-14 12:46:03.612000000,team,,769,"[p_0210c5, p_5969fa, p_56ca3a, p_d306ce, p_d34...",p_5969fa,,0.00
1,r_7dc87606ff728775,2026-11-14 12:46:03.256000000,team,,1133,"[p_adc3e6, p_a9b2bd, p_3c342d, p_dc1c2e, p_021...",,,0.00
2,r_6dba3d12a50e84e4,2026-11-14 12:46:02.937000000,team,,891,"[p_e2454f, p_ac4301, p_a187f7, p_dc1c2e, p_b49...",,,0.00
3,r_47400486faa78044,2026-11-14 12:46:02.577000000,team,,1242,"[p_1036d3, p_ada8ad, p_adc3e6, p_0ceafe, p_e53...",,,0.00
4,r_29e5d31c7b6723ef,2026-11-14 12:46:02.270000000,team,,525,"[p_40ec57, p_36d6fe, p_958ada, p_ceb8c8, p_1bc...",,,0.00
5,r_bdd6381a75c69324,2026-11-14 12:46:01.905000000,team,,881,"[p_1036d3, p_adc3e6, p_dc1c2e, p_0210c5, p_dbc...",,,0.00
6,r_b532fb746d482904,2026-11-14 12:46:01.599000000,team,,167,"[p_2c88d5, p_49d2c3, p_ac4b0d, p_d49ad0, p_95d...",,,0.00
7,r_dfdbe9d7f99af7f0,2026-11-14 12:46:01.243000000,team,,482,"[p_369d20, p_13be1a, p_1036d3, p_5733ef, p_adc...",,,0.00
8,r_08fce0831f6a6aa1,2026-11-14 12:46:00.929000000,team,,841,"[p_aa3b8d, p_1d2fae, p_52e7e7, p_92e7d7, p_a35...",,,0.00
9,r_bc25a5059e64e513,2026-11-14 12:46:00.550000000,team,,644,"[p_61efd3, p_adc3e6, p_68498e, p_f4e307, p_032...",p_032d24,p_032d24,229.99


## 13 · Conversión e ingresos por hora


In [ ]:

sql = f"""
SELECT toStartOfHour(ts::DateTime64(3)) AS hour, served_by::String AS served_by,
 count() AS requests, 100.0 * countIf(ifNull(clicked::String, '') != '') / count() AS ctr_pct,
 100.0 * countIf(ifNull(purchased::String, '') != '') / count() AS conversion_pct,
 sum(revenue_eur::Float64) / count() AS revenue_per_request
FROM {TEAM}.feedback WHERE request_id != '__init__'
AND ts::DateTime64(3) > now() - INTERVAL 24 HOUR
GROUP BY hour, served_by ORDER BY hour DESC, served_by
"""
result = query(sql)


## 14 · Fiabilidad y motivos de fallback


In [ ]:

sql = f"""
SELECT served_by::String AS served_by, fallback_reason::String AS reason,
 count() AS requests, quantile(0.5)(latency_ms::Float64) AS p50_ms,
 quantile(0.95)(latency_ms::Float64) AS p95_ms,
 quantile(0.99)(latency_ms::Float64) AS p99_ms
FROM {TEAM}.feedback WHERE request_id != '__init__'
AND ts::DateTime64(3) > now() - INTERVAL 30 MINUTE
GROUP BY served_by, reason ORDER BY requests DESC, served_by, reason
"""
result = query(sql)


## 15 · Últimas respuestas enviadas
Orden por ID, no por hora: submissions no incluye timestamp en el contrato del reto.

In [ ]:

sql = f"""
SELECT request_id, model_version, items FROM {TEAM}.submissions
WHERE request_id != '__init__' ORDER BY request_id::String DESC LIMIT 50
"""
result = query(sql)


## 16 · Respuestas duplicadas


In [ ]:

sql = f"""
SELECT request_id::String AS request_id, count() AS copies
FROM {TEAM}.submissions WHERE request_id != '__init__'
GROUP BY request_id HAVING copies > 1 ORDER BY copies DESC, request_id LIMIT 50
"""
result = query(sql)


## 17 · Tamaño y unicidad de respuestas
Comprueba longitud y duplicados; no valida disponibilidad histórica.

In [ ]:

sql = f"""
SELECT request_id, model_version, length(CAST(items AS Array(String))) AS n,
 length(arrayDistinct(CAST(items AS Array(String)))) AS distinct_n
FROM {TEAM}.submissions WHERE request_id != '__init__' AND (n != 10 OR distinct_n != 10)
ORDER BY request_id::String DESC LIMIT 50
"""
result = query(sql)


## 18 · Trazar una petición


In [ ]:
request_id = "r_xxxxxx"
sql = f"""
SELECT request_id, ts, served_by, fallback_reason, latency_ms, shown, clicked, purchased
FROM {TEAM}.feedback WHERE request_id::String = {literal(request_id)} LIMIT 10
"""
result = query(sql)


## 19 · Sesgo de posición histórico
Slot 0 significa que el clic no figura en shown; conviene investigarlo.

In [ ]:
start = "2026-11-01"
end = "2026-11-08"
sql = f"""
SELECT indexOf(CAST(shown AS Array(String)), clicked::String) AS slot, count() AS clicks
FROM shop.history_sessions WHERE clicked IS NOT NULL AND clicked::String != ''
AND ts::DateTime64(3) >= {literal(start)} AND ts::DateTime64(3) < {literal(end)}
GROUP BY slot ORDER BY slot
"""
result = query(sql)


## 20 · Auditar propensities de primer slot
Asume arrays numéricos según el contrato. Si el cast falla, usar la siguiente celda para inspeccionar tipos.

In [ ]:
start = "2026-11-01"
end = "2026-11-08"
sql = f"""
SELECT dynamicType(propensity) AS stored_type, count() AS rows,
 countIf(length(CAST(propensity AS Array(Float64))) = 0) AS empty_arrays,
 countIf(NOT isFinite(CAST(propensity AS Array(Float64))[1])
 OR CAST(propensity AS Array(Float64))[1] <= 0
 OR CAST(propensity AS Array(Float64))[1] > 1) AS invalid_first_slot
FROM shop.history_sessions
WHERE ts::DateTime64(3) >= {literal(start)} AND ts::DateTime64(3) < {literal(end)}
GROUP BY stored_type ORDER BY rows DESC
"""
result = query(sql)


## 21 · Tipos reales y valores ausentes


In [ ]:

sql = f"""
SELECT dynamicType(propensity) AS propensity_type, dynamicType(clicked) AS clicked_type,
 count() AS rows, countIf(propensity IS NULL) AS missing_propensity
FROM shop.history_sessions GROUP BY propensity_type, clicked_type
ORDER BY rows DESC, propensity_type, clicked_type
"""
result = query(sql)


## 22 · Propensities crudas para diagnóstico


In [ ]:

sql = f"""
SELECT request_id, propensity, shown, clicked FROM shop.history_sessions
ORDER BY ts::DateTime64(3) DESC, request_id::String LIMIT 50
"""
result = query(sql)


## 23 · Mezcla de tráfico y cold start


In [ ]:

sql = f"""
SELECT context.device::String AS device, context.page::String AS page,
 is_new_user::Bool AS new_user, count() AS requests
FROM shop.requests WHERE ts::DateTime64(3) > now() - INTERVAL 1 HOUR
GROUP BY device, page, new_user ORDER BY requests DESC, device, page, new_user
"""
result = query(sql)


## 24 · Plan de una consulta
Para comparar rendimiento, usa rangos temporales fijos y revisa statistics, no solo el tiempo HTTP.

In [ ]:
product_id = "p_xxxxxx"
sql = f"""
EXPLAIN SELECT product_id, ts, event_type FROM shop.events
WHERE product_id::String = {literal(product_id)} LIMIT 20
"""
result = query(sql)


## Petición libre
Editar SQL directamente. El helper devuelve un DataFrame y muestra estado HTTP, estadísticas y hints. No imprime headers ni la clave. Para comparar ejecuciones guarda el resultado con otro nombre; ninguna celda de arriba depende de `result`.


In [ ]:
sql = "SELECT now() AS server_time"
result = query(sql)
# Exportación local opcional:
# result.to_csv("resultado.csv", index=False)
